# Data Mining - House Price Prediction\n\nNotebook mô phỏng quy trình khai phá dữ liệu: Data Selection, Data Cleaning, Remove Outlier, Data Transformation, Data Mining và Evaluation.

## 1. Data Selection

In [ ]:
import os\nimport warnings\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import MinMaxScaler\nfrom torch.optim import SGD\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom crawling import RAW_OUTPUT, CLEAN_OUTPUT, save_datasets, data_cleaning, data_standard, remove_outlier_zscore\n\nwarnings.filterwarnings('ignore')

In [ ]:
if not os.path.exists(RAW_OUTPUT):\n    save_datasets(max_pages=3, sleep_seconds=0.5)\n\nraw_df = pd.read_csv(RAW_OUTPUT)\nraw_df.head()

## 2. Data Cleaning

In [ ]:
df = data_cleaning(raw_df)\ndf.head()

In [ ]:
df = data_standard(df)\nfor column in ['Diện tích', 'Giá phòng', 'Ngày', 'Tháng', 'Năm']:\n    df[column] = pd.to_numeric(df[column])\n\nprint(df.dtypes)\ndf.head()

## 3. Remove Outlier bằng Z-score

In [ ]:
df_clean = remove_outlier_zscore(df, columns=['Diện tích', 'Giá phòng'], threshold=3)\ndf_clean.to_csv(CLEAN_OUTPUT, index=False, encoding='utf-8-sig')\nprint(df.shape, df_clean.shape)\ndf_clean.head()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))\nz_area = abs((df['Diện tích'] - df['Diện tích'].mean()) / df['Diện tích'].std(ddof=0))\nax[0].hist(z_area)\nax[0].set_title('Dạng phân bố z-score của Diện tích')\nax[1].scatter(df_clean['Diện tích'], df_clean['Giá phòng'], color='red')\nax[1].set_xlabel('Diện tích')\nax[1].set_ylabel('Giá phòng')\nax[1].set_title('Mối tương quan giữa giá phòng và diện tích')\nplt.show()

## 4. Data Transformation

In [ ]:
X = np.array(df_clean.drop(columns=['Giá phòng']))\ny = np.array(df_clean['Giá phòng'])\n\nscaler = MinMaxScaler()\nX = scaler.fit_transform(X)\nX[:5]

## 5. Data Mining & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=30)\nprint(len(X_train[0]))

In [ ]:
def rmse_score(y_true, y_pred):\n    return float(np.sqrt(mean_squared_error(y_true, y_pred)))\n\ndef evaluate(name, y_true, y_pred):\n    return {\n        'Model': name,\n        'MAE': mean_absolute_error(y_true, y_pred),\n        'MSE': mean_squared_error(y_true, y_pred),\n        'RMSE': rmse_score(y_true, y_pred),\n        'R2': r2_score(y_true, y_pred),\n    }\n\nresults = []

### 5.1 Linear Regression

In [ ]:
linear_model = LinearRegression()\nlinear_model.fit(X_train, y_train)\ny_predict = linear_model.predict(X_test)\nprint('Giá nhà được dự đoán là:', y_predict[:20])\nlinear_result = evaluate('Linear Regression', y_test, y_predict)\nresults.append(linear_result)\nprint('Coefficients:', linear_model.coef_)\nprint('Intercept:', linear_model.intercept_)\nlinear_result

### 5.2 Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=150, random_state=30, min_samples_leaf=2)\nrf_model.fit(X_train, y_train)\nrf_predict = rf_model.predict(X_test)\nprint('Giá nhà được dự đoán bởi Random Forest:', rf_predict[:20])\nrf_result = evaluate('Random Forest Regressor', y_test, rf_predict)\nresults.append(rf_result)\nrf_result

### 5.3 XGBoost Regressor

In [ ]:
try:\n    from xgboost import XGBRegressor\n    xgb_model = XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=4, objective='reg:squarederror', random_state=30)\n    xgb_model.fit(X_train, y_train)\n    xgb_predict = xgb_model.predict(X_test)\n    print('Giá nhà được dự đoán bởi XGBoost:', xgb_predict[:20])\n    xgb_result = evaluate('XGBoost Regressor', y_test, xgb_predict)\n    results.append(xgb_result)\n    display(xgb_result)\nexcept ImportError:\n    print('xgboost is not installed')

### 5.4 Neural Network Regression bằng PyTorch

In [ ]:
class NeuralNetworkRegression(nn.Module):\n    def __init__(self, input_size):\n        super().__init__()\n        self.fc1 = nn.Linear(input_size, 32)\n        self.fc2 = nn.Linear(32, 16)\n        self.fc3 = nn.Linear(16, 1)\n\n    def forward(self, x):\n        x = torch.relu(self.fc1(x))\n        x = torch.relu(self.fc2(x))\n        return self.fc3(x)\n\nclass HousePriceDataset(Dataset):\n    def __init__(self, X_data, y_data):\n        self.X_data = torch.FloatTensor(X_data)\n        self.y_data = torch.FloatTensor(y_data).view(-1, 1)\n\n    def __getitem__(self, index):\n        return self.X_data[index], self.y_data[index]\n\n    def __len__(self):\n        return len(self.X_data)\n\ntrain_loader = DataLoader(HousePriceDataset(X_train, y_train), batch_size=128, shuffle=True)\ndevice = 'cuda' if torch.cuda.is_available() else 'cpu'\nnn_model = NeuralNetworkRegression(input_size=X_train.shape[1]).to(device)\ncriterion = nn.MSELoss()\noptimizer = SGD(nn_model.parameters(), lr=0.01)\n\nfor epoch in range(80):\n    running_loss = 0\n    for batch_idx, (samples, labels) in enumerate(train_loader):\n        samples = samples.to(device)\n        labels = labels.to(device)\n        optimizer.zero_grad()\n        outputs = nn_model(samples)\n        loss = criterion(outputs, labels)\n        loss.backward()\n        optimizer.step()\n        running_loss += loss.item()\n    if (epoch + 1) % 20 == 0:\n        print(f'Epoch {epoch + 1}, Loss: {running_loss / (batch_idx + 1):.4f}')\n\nnn_model.eval()\nwith torch.no_grad():\n    pred_nn = nn_model(torch.FloatTensor(X_test).to(device)).cpu().numpy().ravel()\nnn_result = evaluate('Neural Network Regression', y_test, pred_nn)\nresults.append(nn_result)\nnn_result

## 6. So sánh kết quả

In [ ]:
result_df = pd.DataFrame(results).sort_values('RMSE')\nresult_df